# Estrategia V2 — Scraping de Noticias Colombia

## Qué cambia respecto a V1

| | V1 (`scrappers.py`) | V2 (`scrappers_v2.py`) |
|---|---|---|
| Timeout scrapers problemáticos | 300–600 s | **60 s** (falla rápido) |
| Complemento central | Solo si local < 50 arts | **Siempre** si total < 100 arts |
| Respaldo | `eltiempo` + `las2orillas` | **Solo `eltiempo`** |
| Términos con ElTiempo | 5 en paralelo (queue → timeout) | **Secuencial** (todos completan) |
| Salida pkl | `df_corpus_<depto>.pkl` | `df_corpus_v2_<depto>.pkl` |

### Scrapers clasificados
- **Confiables** (timeout 300 s): `elcolombiano`, `elpais`, `diariooccidente`, `eldiario`, `bcnoticias`, `elquindiano`, `diariodelcauca`, `diariodelsur`, `elpilon`, `elmeridiano`, `diariodecasanare`, `miputumayo`
- **Problemáticos** (timeout 60 s): `llanoalmundo`, `lavozdelcinaruco`, `tronchandosinfronteras`, `choco7dias`, `enlacetelevision`, `corrillos`, `portafolio`, `publimetro`

## Opción A — Correr un departamento individual (pruebas)

In [ ]:
import sys, os
# Apuntar al directorio raíz del proyecto (donde están scrappers.py y scrappers_v2.py)
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from scrappers_v2 import scrape_departamento_v2

df = scrape_departamento_v2(
    departamento='Arauca',
    fecha_desde='2023-01-01',
    fecha_hasta='2023-01-31',
)
print(f'\nTotal: {len(df)} artículos')
df[['periodico', 'titulo', 'fecha']].head(10)

## Opción B — Correr un grupo (igual que los grupo_XX_v2.py)

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from scrappers_v2 import scrape_multiples_departamentos_v2

FECHA_DESDE = '2023-01-01'
FECHA_HASTA = '2023-12-31'
DIRECTORIO_SALIDA = os.path.join(ROOT, 'resultados')
os.makedirs(DIRECTORIO_SALIDA, exist_ok=True)

# Ejemplo: Grupo 3 — Caldas, Meta, Bolívar
scrape_multiples_departamentos_v2(
    departamentos=['Caldas', 'Meta', 'Bolívar'],
    fecha_desde=FECHA_DESDE,
    fecha_hasta=FECHA_HASTA,
    directorio_salida=DIRECTORIO_SALIDA,
)

## Opción C — Correr desde terminal (recomendado para el corpus completo)

Cada grupo puede correr en una ventana/terminal separada **en paralelo**:

```bash
# Desde la carpeta v2/
cd v2

# Ventana 1
py -3.13 -X utf8 grupo_01_v2.py   # Antioquia · Chocó · Vichada

# Ventana 2
py -3.13 -X utf8 grupo_02_v2.py   # Valle del Cauca · Arauca · Atlántico

# Ventana 3
py -3.13 -X utf8 grupo_03_v2.py   # Caldas · Meta · Bolívar

# ... etc hasta grupo_11_v2.py
```

> Los pkl se guardan en `../resultados/df_corpus_v2_<departamento>.pkl`

## Monitorear progreso

In [ ]:
import glob, pickle, os
from datetime import datetime

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RESULTADOS = os.path.join(ROOT, 'resultados')

# 32 departamentos de Colombia
DEPARTAMENTOS = [
    'Antioquia', 'Choco', 'Vichada',
    'Valle_del_Cauca', 'Arauca', 'Atlantico',
    'Caldas', 'Meta', 'Bolivar',
    'Risaralda', 'Caqueta', 'Putumayo',
    'Quindio', 'Guaviare', 'Amazonas',
    'Santander', 'Cesar', 'Boyaca',
    'Norte_de_Santander', 'Magdalena', 'Guainia',
    'Cundinamarca', 'La_Guajira', 'Cauca',
    'Cordoba', 'Narino', 'Vaupes',
    'Sucre', 'Huila', 'San_Andres_y_Providencia',
    'Casanare', 'Tolima',
]

print(f"{'Departamento':<30} {'Arts':>6}  {'MB':>5}  {'Actualizado'}")
print('-' * 65)
completados = 0
for dep in DEPARTAMENTOS:
    patron = os.path.join(RESULTADOS, f'df_corpus_v2_{dep.lower()}.pkl')
    archivos = glob.glob(patron)
    if archivos:
        f = archivos[0]
        size_mb = os.path.getsize(f) / 1e6
        mtime = datetime.fromtimestamp(os.path.getmtime(f)).strftime('%m-%d %H:%M')
        try:
            df = pickle.load(open(f, 'rb'))
            n = len(df)
        except Exception:
            n = '?'
        print(f'  {dep:<28} {str(n):>6}  {size_mb:>5.1f}  {mtime}')
        completados += 1
    else:
        print(f'  {dep:<28} {"pendiente":>6}')

print(f'\n{completados}/32 departamentos completados')

## Cargar y consolidar corpus V2

In [ ]:
import glob, pickle, pandas as pd, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RESULTADOS = os.path.join(ROOT, 'resultados')

archivos = sorted(glob.glob(os.path.join(RESULTADOS, 'df_corpus_v2_*.pkl')))
print(f'Archivos encontrados: {len(archivos)}')

corpus = pd.concat(
    [pickle.load(open(f, 'rb')) for f in archivos],
    ignore_index=True
)

print(f'Total artículos: {len(corpus):,}')
print(f'Departamentos: {corpus["departamento"].nunique()}')
print(f'Periódicos: {corpus["periodico"].nunique()}')
print(f'Rango fechas: {corpus["fecha"].min()} — {corpus["fecha"].max()}')
print()
print(corpus.groupby('departamento').size().sort_values(ascending=False).to_string())